In [0]:
dbutils.widgets.removeAll()

In [0]:
import logging
from datetime import datetime
import os

# Create logs directory dynamically if it doesn't exist
try:
    notebook_dir = os.getcwd()
    project_root = os.path.abspath(os.path.join(notebook_dir, "../../.."))
except Exception:
    project_root = "/tmp"

log_dir = os.path.join(project_root, "logs")
os.makedirs(log_dir, exist_ok=True)

# Create log filename with timestamp
log_filename = os.path.join(log_dir, f"provider_hierarchy_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log")

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_filename),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger(__name__)
logger.info("=" * 60)
logger.info("Provider Hierarchy Silver Layer Processing Started")
logger.info(f"Log file: {log_filename}")
logger.info("=" * 60)

In [0]:
try:
    dbutils.widgets.text("ClientContainer", "new", "Client Container / Catalog Name")
    client_container = dbutils.widgets.get("ClientContainer").strip()
except Exception:
    client_container = "new"

# Wrap catalog name in backticks for numerical safety (e.g. `274`)
safe_catalog = f"`{client_container}`"

sourcePath = f"/Volumes/{client_container}/bronze/processed_parquet/provider_hierarchy"
silverHierarchyTable = f"{safe_catalog}.silver.provider_hierarchy"
affiliationPath = f"{safe_catalog}.silver.ref_provider_affiliation"
credentialingPath = f"{safe_catalog}.silver.ref_credentialing"

print("="*60)
print("CONFIGURATION")
print("="*60)
print(f"Source Path: {sourcePath}")
print(f"Silver Hierarchy Table: {silverHierarchyTable}")
print("="*60)

logger.info("Configuration loaded:")
logger.info(f"  Source Path: {sourcePath}")
logger.info(f"  Silver Hierarchy Table: {silverHierarchyTable}")

In [0]:
def table_exists(tableToCheck):
    """Check if a table or path exists and can be read"""
    try:
        if '/' in tableToCheck:
            spark.read.format("parquet").load(tableToCheck)
        else:
            spark.table(tableToCheck)
        logger.info(f"Data check: {tableToCheck} - Data exists")
        return True
    except Exception as e:
        logger.warning(f"Data check: {tableToCheck} - Data does not exist or is inaccessible: {str(e)}")
        return False

In [0]:
from pyspark.sql.functions import date_format, sha2, concat_ws, col, upper, trim, coalesce, lit

In [0]:
logger.info("MAIN EXECUTION STARTED")

from pyspark.sql.types import StructType, StructField, StringType

if table_exists(sourcePath):
    print("Loading Reference Tables...")
    if table_exists(affiliationPath):
        if '/' in affiliationPath:
            spark.read.format("parquet").load(affiliationPath).createOrReplaceTempView("ref_affiliation")
        else:
            spark.table(affiliationPath).createOrReplaceTempView("ref_affiliation")
    else:
        spark.createDataFrame([], StructType([
            StructField("Tier1BillingProviderTIN", StringType(), True),
            StructField("Tier2ID", StringType(), True),
            StructField("Tier2Description", StringType(), True),
            StructField("Tier3ID", StringType(), True),
            StructField("Tier3Description", StringType(), True)
        ])).createOrReplaceTempView("ref_affiliation")
        
    if table_exists(credentialingPath):
        if '/' in credentialingPath:
            spark.read.format("parquet").load(credentialingPath).createOrReplaceTempView("ref_credentialing")
        else:
            spark.table(credentialingPath).createOrReplaceTempView("ref_credentialing")
    else:
        spark.createDataFrame([], StructType([
            StructField("ProviderID", StringType(), True),
            StructField("DONOTCHASE", StringType(), True)
        ])).createOrReplaceTempView("ref_credentialing")

    print("Loading Bronze Provider Hierarchy data...")
    dfHierarchySrc = spark.read.format("parquet").load(sourcePath)
    
    # Apply joins and coalescing
    df_aff = spark.table("ref_affiliation")
    df_cred = spark.table("ref_credentialing")
    
    # Make sure we don't duplicate columns if they exist in source
    cols_to_check = ["DONOTCHASE", "TIER2ID", "TIER2DESC", "TIER3ID", "TIER3DESC", "TIER2ADDRESS1", "TIER2CITY", "TIER2STATE", "TIER2ZIP"]
    for c in cols_to_check:
        if c not in dfHierarchySrc.columns:
            dfHierarchySrc = dfHierarchySrc.withColumn(c, lit(None))
            
    dfHierarchySrc.createOrReplaceTempView("provider_hierarchy")
    
    hier_sql = """
    SELECT 
        h.* EXCEPT (DONOTCHASE, TIER2ID, TIER2DESC, TIER3ID, TIER3DESC, TIER2ADDRESS1, TIER2CITY, TIER2STATE, TIER2ZIP),
        coalesce(c.DONOTCHASE, h.DONOTCHASE, 'N') AS DONOTCHASE,
        coalesce(h.TIER2ID, aff.Tier2ID) AS TIER2ID,
        coalesce(h.TIER2DESC, aff.Tier2Description) AS TIER2DESC,
        coalesce(h.TIER3ID, aff.Tier3ID) AS TIER3ID,
        coalesce(h.TIER3DESC, aff.Tier3Description) AS TIER3DESC,
        coalesce(h.TIER2ADDRESS1, h.LOCATIONADDRESS1) AS TIER2ADDRESS1,
        coalesce(h.TIER2CITY, h.LOCATIONCITY) AS TIER2CITY,
        coalesce(h.TIER2STATE, h.LOCATIONSTATE) AS TIER2STATE,
        coalesce(h.TIER2ZIP, h.LOCATIONZIP) AS TIER2ZIP
    FROM provider_hierarchy h
    LEFT JOIN ref_affiliation aff ON h.LOCATIONTIN = aff.Tier1BillingProviderTIN
    LEFT JOIN ref_credentialing c ON h.PROVIDERID = c.ProviderID
    """
    dfHierarchyJoined = spark.sql(hier_sql)
    
    print("Applying cleaning and normalization transformations...")
    # Normalise states, trim strings, handle potential nulls
    dfHierarchy = dfHierarchyJoined.distinct() \
        .withColumn("PROVIDERID", upper(trim(col("PROVIDERID")))) \
        .withColumn("PROVIDERLASTNAME", upper(trim(col("PROVIDERLASTNAME")))) \
        .withColumn("PROVIDERNPI", upper(trim(col("PROVIDERNPI")))) \
        .withColumn("LOCATIONGROUPID", upper(trim(col("LOCATIONGROUPID")))) \
        .withColumn("LOCATIONIDTYPE", upper(trim(col("LOCATIONIDTYPE")))) \
        .withColumn("LOCATIONID", upper(trim(col("LOCATIONID")))) \
        .withColumn("LOCATIONADDRESS1", upper(trim(col("LOCATIONADDRESS1")))) \
        .withColumn("LOCATIONCITY", upper(trim(col("LOCATIONCITY")))) \
        .withColumn("LOCATIONSTATE", upper(trim(col("LOCATIONSTATE")))) \
        .withColumn("LOCATIONZIP", upper(trim(col("LOCATIONZIP"))))
        
    # Generate unique HashKey for tracking changes
    dfHierarchy = dfHierarchy.withColumn(
        "HashKey", 
        sha2(concat_ws("||", *dfHierarchy.columns), 256)
    )
    
    recordCount = dfHierarchy.count()
    print(f"Found {recordCount} records to write")
    
    if recordCount > 0:
        print(f"Writing to Silver Hierarchy Table: {silverHierarchyTable}")
        spark.sql(f"DROP TABLE IF EXISTS {silverHierarchyTable}")
        dfHierarchy.write.format("delta").mode("overwrite").saveAsTable(silverHierarchyTable)
        print("Silver Provider Hierarchy table created successfully!")
    else:
        print("No records found to write.")
else:
    print("Bronze source path does not exist. Skipping execution.")
    logger.error("Source path missing")